# Generator

Um `Generator` basicamente gera valores sob-demanda, ele não carrega em memória ele gera quando necessario.

In [84]:
from typing import Generator
from itertools import cycle

- `Generator Functions`: São iguais a *funções* mas usam **yield** ao invés de **return**;
- `Generator Expressions`: São como list/set/dicts comprehensions mas usam `()` ao invés de `[] / {}`

In [2]:
gen_expr = (i for i in range(10) if i % 2 == 0)

In [3]:
for number in gen_expr:
    print(number)

0
2
4
6
8


A notação de tipo de um `Generator` segue a seguinte estrutura:
```python
from typing import Generator
Generator[tipo yield, tipo send, tipo return]
```

In [4]:
def generator() -> Generator[int, None, None]:
    i = 0
    while True:
        yield i
        i += 1

In [5]:
c = generator()

In [6]:
for i in cycle(c):
    print(i)
    if i == 10:
        break

0
1
2
3
4
5
6
7
8
9
10


In [7]:
next(c)

11

Podemos criar um gerador infinito de numeros `pares/impares`.

In [12]:
from typing import Literal

def generator(_type: Literal['even', 'odd']):
    n = 0 if _type == 'even' else 1
    while True:
        yield n
        n += 2

In [13]:
even_gen = generator('even')

odd_gen = generator('odd')

In [14]:
for number in even_gen:
    print(number)
    if number == 10:
        break

0
2
4
6
8
10


In [15]:
for number in odd_gen:
    print(number)
    if number == 11:
        break

1
3
5
7
9
11


Podemos criar um `generator` que gera números da sequencia de Fibonacci.
$F_n = F_{n-1} + F_{n-2}$

In [54]:
def fibonacci_generator():
    f_0, f_1 = 0, 1
    while True:
        f = f_0 + f_1
        yield f
        f_0, f_1 = f_1, f

In [55]:
fib_gen = fibonacci_generator()

In [56]:
for fib in fib_gen:
    print(fib)
    if fib == 21:
        break

1
2
3
5
8
13
21


Podemos fazer generator que faz a soma de sequências infinitas.

$$e = \sum_{n=0}^{\infty} \frac{1}{n!}$$

In [58]:
from math import factorial

def euler():
    n = 0
    e = 0
    while True:
        e += 1 / factorial(n)
        yield e
        n += 1

In [60]:
euler_gen = euler()
for n in euler_gen:
    print(f'{n:.5f}')
    if n >= 2.716:
        break

1.00000
2.00000
2.50000
2.66667
2.70833
2.71667


Podemos chegar perto do número (π).

$$
\frac{\pi}{4} = \sum_{n=0}^{\infty} \frac{(-1)^n}{2n+1}
$$

In [69]:
def pi():
    n = 0
    pi = 0
    while True:
        pi += ((-1) ** n ) / (2 * n + 1)
        yield pi * 4
        n += 1

In [78]:
import math

PI = math.pi

pi_gen = pi()
for i, n in enumerate(pi_gen):
    error = abs(n - PI)
    if i % 10 == 0:
        print(f'π gen: {n:.4f} | π real: {PI:.4f} |error: {error:.4f}')
    if error <= 0.01:
        break

π gen: 4.0000 | π real: 3.1416 |error: 0.8584
π gen: 3.2323 | π real: 3.1416 |error: 0.0907
π gen: 3.1892 | π real: 3.1416 |error: 0.0476
π gen: 3.1738 | π real: 3.1416 |error: 0.0322
π gen: 3.1660 | π real: 3.1416 |error: 0.0244
π gen: 3.1612 | π real: 3.1416 |error: 0.0196
π gen: 3.1580 | π real: 3.1416 |error: 0.0164
π gen: 3.1557 | π real: 3.1416 |error: 0.0141
π gen: 3.1539 | π real: 3.1416 |error: 0.0123
π gen: 3.1526 | π real: 3.1416 |error: 0.0110


## Corrotina
`Corrotinas` são funções que podem pausar a execução, e depois voltar de onde pararam. Ela tem algumas peculiaridades:
- Mantém o estado interno;
- Não termina assim que chamada;
- Pode interagir com quem a chama.

```python
x = yield  # aqui ocorre a pausa do programa e só continua quando enviamos (send) o valor.
```

In [79]:
def coroutine() -> Generator[int, None, None]:
    x = yield 'esperando valor'
    print(f'recebi: {x}')
    y = yield 'Esperando segundo valor'
    print(f'recebi: {y}')

In [80]:
c = coroutine()

In [81]:
next(c)  # inicia a corrotina

'esperando valor'

In [82]:
c.send(10)

recebi: 10


'Esperando segundo valor'

In [93]:
def worker():
    total = 0
    while True:
        x = yield total
        total += x

        if total > 100:
            yield "Worker finalizado"
            return

In [94]:
w = worker()

In [95]:
next(w)  # inicia a corrotina

0

In [96]:
print(w.send(10))
print(w.send(20))
print(w.send(30))
print(w.send(40))
print(w.send(50))

10
30
60
100
Worker finalizado


Esse **comportamento** é muito interessante pois ele abre as portas para `async def` e `await` pois agora nossas funções *"possuem estado interno"* elas podem começar a executação e `await - (aguardar)` outras coisas liberando o processamento e depois retomar para a sua execução até que ela finalize.